In [3]:
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
         .appName("afrimarket-eda")
         .config("spark.driver.memory", "4g")
         .getOrCreate())

df = spark.read.csv("data/afrimarket_transactions.csv", header=True, inferSchema=True)
df.cache()

n_rows = df.count()
print(f"Rows: {n_rows:,}   Columns: {len(df.columns)}")

fraud_rate = df.agg(F.mean("isFraud")).first()[0]
print(f"Fraud rate: {fraud_rate:.2%}")

dup_ids = n_rows - df.dropDuplicates(["TransactionID"]).count()
dup_rows = n_rows - df.dropDuplicates().count()
print(f"Duplicate TransactionIDs: {dup_ids}   Fully duplicated rows: {dup_rows}")

print("Column groups:")
for prefix in ["card", "addr", "dist", "C", "D", "M", "V", "id_"]:
    cols = [c for c in df.columns if c.startswith(prefix) and
            (prefix in ["card", "addr", "dist", "id_"] or c[len(prefix):].isdigit())]
    print(f"  {prefix:<5} {len(cols)} columns")

Rows: 30,000   Columns: 434
Fraud rate: 3.50%
Duplicate TransactionIDs: 0   Fully duplicated rows: 0
Column groups:
  card  6 columns
  addr  2 columns
  dist  2 columns
  C     14 columns
  D     15 columns
  M     9 columns
  V     339 columns
  id_   38 columns


In [4]:
import pandas as pd

used_by_abdullahi = ["TransactionAmt", "card1", "card2", "card3", "card5", "addr1", "dist1",
                     "C1", "C2", "C5", "C6", "C9", "C11", "C13", "C14", "D1", "D4", "D10", "D15",
                     "ProductCD", "card4", "card6", "P_emaildomain", "R_emaildomain", "DeviceType"]

# Share of missing values for every column, in ONE pass over the data
missing = df.select([F.mean(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).toPandas().T
missing.columns = ["missing_pct"]
missing["missing_pct"] = (missing["missing_pct"] * 100).round(1)

def group(c):
    for p in ["card", "addr", "dist", "id_"]:
        if c.startswith(p): return p
    for p in ["C", "D", "M", "V"]:
        if c.startswith(p) and c[1:].isdigit(): return p
    return "other"

missing["group"] = [group(c) for c in missing.index]
missing["used"] = missing.index.isin(used_by_abdullahi)
missing["bucket"] = pd.cut(missing["missing_pct"], [-1, 0, 20, 50, 90, 100],
                           labels=["0%", "1-20%", "20-50%", "50-90%", "90-100%"])

print("Number of columns by group and by % missing:")
print(pd.crosstab(missing["group"], missing["bucket"]))

print("\nAll non-V columns (sorted by % missing):")
pd.set_option("display.max_rows", 200)
missing[missing["group"] != "V"].sort_values("missing_pct")

Number of columns by group and by % missing:
bucket  0%  1-20%  20-50%  50-90%  90-100%
group                                     
C       14      0       0       0        0
D        0      3       4       7        1
M        0      0       5       4        0
V       75     76      29     159        0
addr     0      2       0       0        0
card     1      5       0       0        0
dist     0      0       0       1        1
id_      0      0       0      28       10
other    5      1       0       3        0

All non-V columns (sorted by % missing):


,missing_pct,group,used,bucket
TransactionID,0.0,other,False,0%
C14,0.0,C,True,0%
C13,0.0,C,True,0%
C12,0.0,C,False,0%
C11,0.0,C,True,0%
C10,0.0,C,False,0%
C9,0.0,C,True,0%
C8,0.0,C,False,0%
C7,0.0,C,False,0%
C6,0.0,C,True,0%


In [5]:
# Non-V columns that are partly missing (but not 90%+ empty)
candidates = missing[(missing["group"] != "V") &
                     (missing["missing_pct"] > 0) &
                     (missing["missing_pct"] < 90)].index.tolist()

aggs = []
for c in candidates:
    is_null = F.col(c).isNull()
    aggs += [F.sum(is_null.cast("int")).alias(f"{c}__n_missing"),
             F.sum(F.when(is_null, F.col("isFraud"))).alias(f"{c}__fraud_missing"),
             F.sum(F.when(~is_null, F.col("isFraud"))).alias(f"{c}__fraud_present")]
row = df.agg(*aggs).first().asDict()

rows = []
for c in candidates:
    n_miss = row[f"{c}__n_missing"]
    n_pres = n_rows - n_miss
    rate_miss = (row[f"{c}__fraud_missing"] or 0) / n_miss if n_miss else None
    rate_pres = (row[f"{c}__fraud_present"] or 0) / n_pres if n_pres else None
    rows.append({"column": c,
                 "missing_pct": missing.loc[c, "missing_pct"],
                 "used": missing.loc[c, "used"],
                 "fraud_if_missing_%": round(rate_miss * 100, 2),
                 "fraud_if_present_%": round(rate_pres * 100, 2)})

signal = pd.DataFrame(rows)
signal["ratio"] = (signal["fraud_if_present_%"] / signal["fraud_if_missing_%"]).round(2)
signal.sort_values("ratio", ascending=False)

,column,missing_pct,used,fraud_if_missing_%,fraud_if_present_%,ratio
40,id_10,87.0,False,2.39,10.92,4.57
39,id_09,87.0,False,2.39,10.92,4.57
17,D9,87.0,False,2.39,10.92,4.57
16,D8,87.0,False,2.39,10.92,4.57
20,D12,88.8,False,2.50,11.40,4.56
22,D14,89.4,False,2.55,11.49,4.51
36,id_04,88.5,False,2.52,11.04,4.38
35,id_03,88.5,False,2.52,11.04,4.38
15,D6,87.4,False,2.49,10.48,4.21
9,R_emaildomain,76.3,True,2.02,8.27,4.09


In [6]:
# Training period only: the oldest 70%, the same as the training script
cut = df.approxQuantile("TransactionDT", [0.70], 0.0001)[0]
train_df = df.filter(F.col("TransactionDT") <= cut).cache()
print(f"Training rows: {train_df.count():,}\n")

# 1) Fraud rate for each value of the M columns (T, F, missing...)
for c in [f"M{i}" for i in range(1, 10)]:
    t = (train_df.groupBy(F.coalesce(F.col(c).cast("string"), F.lit("missing")).alias("value"))
         .agg(F.count("*").alias("rows"), F.round(F.mean("isFraud") * 100, 2).alias("fraud_%"))
         .orderBy("value").toPandas())
    print(c, " | ".join(f"{r.value}: {r['fraud_%']}% ({r.rows})" for _, r in t.iterrows()))

# 2) Are the unused C columns just copies of the used ones?
c_used = ["C1", "C2", "C5", "C6", "C9", "C11", "C13", "C14"]
c_unused = ["C3", "C4", "C7", "C8", "C10", "C12"]
corr = train_df.select(c_used + c_unused).toPandas().corr()
print("\nHighest correlation of each unused C column with a used one:")
for c in c_unused:
    best = corr.loc[c, c_used].abs().sort_values(ascending=False)
    print(f"  {c}: {best.index[0]} (r = {best.iloc[0]:.2f})")

Training rows: 20,999

M1 F: 0.0% (4) | T: 1.81% (9665) | missing: 4.82% (11330)
M2 F: 2.55% (1057) | T: 1.72% (8612) | missing: 4.82% (11330)
M3 F: 2.71% (2103) | T: 1.56% (7566) | missing: 4.82% (11330)
M4 M0: 3.56% (6835) | M1: 2.83% (1805) | M2: 9.58% (2214) | missing: 2.12% (10145)
M5 F: 2.23% (4529) | T: 3.75% (3763) | missing: 3.77% (12707)
M6 F: 2.19% (7750) | T: 1.64% (6688) | missing: 6.72% (6561)
M7 F: 1.66% (5954) | T: 2.38% (923) | missing: 4.25% (14122)
M8 F: 1.71% (4375) | T: 1.84% (2502) | missing: 4.25% (14122)
M9 F: 2.74% (1058) | T: 1.58% (5819) | missing: 4.25% (14122)

Highest correlation of each unused C column with a used one:
  C3: C9 (r = 0.01)
  C4: C11 (r = 0.98)
  C7: C2 (r = 0.96)
  C8: C2 (r = 0.98)
  C10: C2 (r = 0.98)
  C12: C2 (r = 0.96)


In [7]:
for c in ["P_emaildomain", "R_emaildomain"]:
    n_values = train_df.select(c).distinct().count()
    print(f"\n{c}: {n_values} different values. Top 15 by number of rows:")
    (train_df.groupBy(c)
     .agg(F.count("*").alias("rows"), F.round(F.mean("isFraud") * 100, 2).alias("fraud_%"))
     .orderBy(F.desc("rows")).show(15, truncate=False))

# Does the purchaser's email domain match the recipient's?
match = (F.when(F.col("R_emaildomain").isNull(), "R missing")
          .when(F.col("P_emaildomain").isNull(), "P missing")
          .when(F.col("P_emaildomain") == F.col("R_emaildomain"), "same")
          .otherwise("different"))
print("Purchaser vs recipient email domain:")
(train_df.groupBy(match.alias("email_match"))
 .agg(F.count("*").alias("rows"), F.round(F.mean("isFraud") * 100, 2).alias("fraud_%"))
 .orderBy("email_match").show())


P_emaildomain: 60 different values. Top 15 by number of rows:
+-------------+----+-------+
|P_emaildomain|rows|fraud_%|
+-------------+----+-------+
|gmail.com    |8083|4.03   |
|yahoo.com    |3552|2.65   |
|NULL         |3336|3.27   |
|hotmail.com  |1742|4.99   |
|anonymous.com|1344|2.6    |
|aol.com      |1009|1.78   |
|comcast.net  |265 |2.26   |
|icloud.com   |180 |3.33   |
|outlook.com  |170 |5.88   |
|msn.com      |153 |1.31   |
|att.net      |132 |1.52   |
|live.com     |119 |1.68   |
|verizon.net  |88  |0.0    |
|sbcglobal.net|85  |1.18   |
|ymail.com    |84  |2.38   |
+-------------+----+-------+
only showing top 15 rows


R_emaildomain: 55 different values. Top 15 by number of rows:
+-------------+-----+-------+
|R_emaildomain|rows |fraud_%|
+-------------+-----+-------+
|NULL         |15668|1.96   |
|gmail.com    |2131 |10.75  |
|hotmail.com  |1105 |7.51   |
|anonymous.com|855  |3.16   |
|yahoo.com    |446  |8.3    |
|aol.com      |154  |6.49   |
|comcast.net  |81   |0.0   

In [9]:
import importlib, features
importlib.reload(features)

out = features.add_features(features.add_uid(features.clean(train_df)))
print("Model columns:", len(features.NUMERIC) + len(features.CATEGORICAL))
for c in ["email_match", "P_provider", "R_provider"]:
    (out.groupBy(c).agg(F.count("*").alias("rows"),
                        F.round(F.mean("isFraud") * 100, 2).alias("fraud_%"))
     .orderBy(F.desc("rows")).show(truncate=False))

Model columns: 42
+-----------+-----+-------+
|email_match|rows |fraud_%|
+-----------+-----+-------+
|R_missing  |15668|1.96   |
|same       |4003 |8.79   |
|different  |981  |3.06   |
|P_missing  |347  |9.22   |
+-----------+-----+-------+

+----------+----+-------+
|P_provider|rows|fraud_%|
+----------+----+-------+
|google    |8097|4.04   |
|yahoo     |3734|2.6    |
|missing   |3336|3.27   |
|microsoft |2249|4.71   |
|anonymous |1344|2.6    |
|aol       |1024|1.95   |
|isp       |841 |1.66   |
|apple     |254 |2.36   |
|other     |120 |5.83   |
+----------+----+-------+

+----------+-----+-------+
|R_provider|rows |fraud_%|
+----------+-----+-------+
|missing   |15668|1.96   |
|google    |2133 |10.74  |
|microsoft |1319 |7.51   |
|anonymous |855  |3.16   |
|yahoo     |526  |7.41   |
|isp       |211  |0.47   |
|aol       |155  |6.45   |
|apple     |94   |7.45   |
|other     |38   |5.26   |
+----------+-----+-------+



In [10]:
spark.stop()